In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [3]:
df = pd.read_csv(r"C:\Users\Aditya P J\Documents\Competition\Satria Data 2025\data\data_train_clean.csv")
df.head()

,id,video,emotion
0,1,https://www.instagram.com/reel/DNKcHgdA-d1/?ig...,Surprise
1,2,https://www.instagram.com/reel/DNHwrh2gnBm/?ig...,Surprise
2,3,https://www.instagram.com/reel/DM7QsjnRCoa/?ig...,Surprise
3,4,https://www.instagram.com/reel/DNBBEt6Paxj/?ig...,Surprise
4,5,https://www.instagram.com/reel/DMz13fQzZsN/?ig...,Proud


In [ ]:
unique_emotion = df['emotion'].unique()
print(unique_emotion)

In [353]:
import pandas as pd
import os
import yt_dlp
import time
from pathlib import Path

class SimpleVideoDownloader:
    def __init__(self, output_dir="downloaded_videos"):
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
    
    def download_video(self, url, video_id, emotion):
        """Download video saja menggunakan yt-dlp dengan cookie authentication"""
        try:
            # Nama file output
            filename = f"{video_id}_{emotion}"
            
            # Konfigurasi yt-dlp - FOKUS HANYA VIDEO
            ydl_opts = {
                'outtmpl': os.path.join(self.output_dir, f'{filename}.%(ext)s'),
                'format': 'best',  # Video quality terbaik <= 720p
                # Alternative browsers:
                # 'cookiesfrombrowser': ('firefox',),
                # 'cookiesfrombrowser': ('safari',),
                
                # Headers untuk bypass detection
                'http_headers': {
                    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
                },
                
                # Retry settings
                'retries': 3,
                'fragment_retries': 3,
                
                # Sleep to avoid rate limiting
                'sleep_interval': 2,
                'max_sleep_interval': 5,
                
                # Suppress output untuk cleaner logs
                'quiet': False,
                'no_warnings': False,
            }
            
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                # Download video
                ydl.download([url])
                
                # Cari file yang berhasil didownload
                for ext in ['mp4', 'webm', 'mkv']:
                    potential_file = os.path.join(self.output_dir, f"{filename}.{ext}")
                    if os.path.exists(potential_file):
                        return potential_file
                
                return None
                
        except Exception as e:
            print(f"❌ Error downloading video {video_id}: {str(e)}")
            return None
    
    def process_csv_batch(self, df, batch_num):
        """Proses satu batch video"""
        results = []
        total_videos = len(df)
        
        print(f"\n🔄 Processing Batch {batch_num} - {total_videos} videos")
        print("="*50)
        
        for index, row in df.iterrows():
            video_id = row['id']
            url = row['video']
            emotion = row['emotion']
            
            print(f"📥 [{index+1}/{total_videos}] Downloading video {video_id} ({emotion})")
            
            # Download video
            video_path = self.download_video(url, video_id, emotion)
            
            if video_path and os.path.exists(video_path):
                file_size = os.path.getsize(video_path) / (1024*1024)  # MB
                print(f"✅ Success: {os.path.basename(video_path)} ({file_size:.1f} MB)")
                
                result = {
                    'id': video_id,
                    'video_file': os.path.basename(video_path),
                    'video_path': video_path,
                    'emotion': emotion,
                    'original_url': url,
                    'status': 'success',
                    'file_size_mb': round(file_size, 1)
                }
            else:
                print(f"❌ Failed: Could not download video {video_id}")
                result = {
                    'id': video_id,
                    'video_file': None,
                    'video_path': None,
                    'emotion': emotion,
                    'original_url': url,
                    'status': 'failed',
                    'file_size_mb': 0
                }
            
            results.append(result)
            
            # Delay antar download
            print(f"⏳ Waiting 3 seconds...")
            time.sleep(3)
        
        return pd.DataFrame(results)

def download_instagram_videos(csv_file_path, batch_size=25, start_from=0):
    """
    Main function untuk download video Instagram
    
    Args:
        csv_file_path (str): Path ke file CSV
        batch_size (int): Jumlah video per batch
        start_from (int): Index mulai (untuk resume)
    """
    
    print("🚀 SIMPLE INSTAGRAM VIDEO DOWNLOADER")
    print("="*50)
    
    try:
        # Baca CSV
        print(f"📖 Reading CSV: {csv_file_path}")
        df = pd.read_csv(csv_file_path)
        
        print(f"📊 Found columns: {df.columns.tolist()}")
        print(f"📈 Total rows: {len(df)}")
        print(f"🎯 Sample data:")
        print(df.head())
        
        # Validasi kolom
        required_cols = ['id', 'video', 'emotion']
        missing_cols = [col for col in required_cols if col not in df.columns]
        if missing_cols:
            print(f"❌ Missing columns: {missing_cols}")
            return
        
        # Filter data untuk diproses
        df_to_process = df.iloc[start_from:].reset_index(drop=True)
        print(f"\n🎯 Processing {len(df_to_process)} rows starting from index {start_from}")
        
        # Inisialisasi downloader
        downloader = SimpleVideoDownloader()
        
        # Process in batches
        all_results = []
        
        for batch_start in range(0, len(df_to_process), batch_size):
            batch_end = min(batch_start + batch_size, len(df_to_process))
            batch_df = df_to_process.iloc[batch_start:batch_end].reset_index(drop=True)
            
            batch_num = (batch_start // batch_size) + 1
            
            # Proses batch
            batch_results = downloader.process_csv_batch(batch_df, batch_num)
            all_results.append(batch_results)
            
            # Simpan hasil batch
            batch_filename = f'batch_videos_{batch_num}.csv'
            batch_results.to_csv(batch_filename, index=False)
            
            # Summary batch
            successful = len(batch_results[batch_results['status'] == 'success'])
            total_size = batch_results['file_size_mb'].sum()
            
            print(f"\n📊 Batch {batch_num} Summary:")
            print(f"   ✅ Downloaded: {successful}/{len(batch_df)} videos")
            print(f"   📊 Success rate: {successful/len(batch_df)*100:.1f}%")
            print(f"   💾 Total size: {total_size:.1f} MB")
            print(f"   💾 Saved to: {batch_filename}")
            
            # Break jika ini batch terakhir
            if batch_end >= len(df_to_process):
                break
                
        # Gabungkan semua hasil
        if all_results:
            final_results = pd.concat(all_results, ignore_index=True)
            
            # Simpan hasil final
            final_filename = f'final_videos_{start_from}_{start_from + len(df_to_process) - 1}.csv'
            final_results.to_csv(final_filename, index=False)
            
            # Summary keseluruhan
            total_successful = len(final_results[final_results['status'] == 'success'])
            total_size = final_results['file_size_mb'].sum()
            
            print(f"\n🎉 FINAL SUMMARY")
            print("="*50)
            print(f"📊 Total processed: {len(final_results)} videos")
            print(f"✅ Successfully downloaded: {total_successful} videos")
            print(f"📈 Overall success rate: {total_successful/len(final_results)*100:.1f}%")
            print(f"💾 Total downloaded size: {total_size:.1f} MB")
            print(f"📁 Videos saved in: downloaded_videos/")
            print(f"📄 Results saved to: {final_filename}")
            
            # Breakdown per emotion
            print(f"\n📈 Breakdown by Emotion:")
            emotion_summary = final_results.groupby('emotion').agg({
                'id': 'count',
                'status': lambda x: (x == 'success').sum(),
                'file_size_mb': 'sum'
            }).rename(columns={
                'id': 'total', 
                'status': 'downloaded',
                'file_size_mb': 'total_size_mb'
            })
            emotion_summary['success_rate_%'] = (emotion_summary['downloaded'] / emotion_summary['total'] * 100).round(1)
            print(emotion_summary)
            
            return final_results
        
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        return None

# CARA PENGGUNAAN
if __name__ == "__main__":
    # SETUP - Ganti dengan path file CSV Anda
    csv_file = r"C:\Users\Aditya P J\Documents\Competition\Satria Data 2025\data\data_train_clean.csv"
    
    print("🔧 SETUP INSTRUCTIONS:")
    print("1. Make sure you're logged into Instagram in Chrome browser")
    print("2. Install required packages: pip install yt-dlp pandas")
    print("3. The script will use cookies from Chrome automatically")
    print("\n" + "="*50)
    
    # Konfirmasi untuk melanjutkan
    confirm = input("Ready to start downloading? (y/n): ").lower().strip()
    
    if confirm == 'y':
        # JALANKAN DOWNLOAD
        results = download_instagram_videos(
            csv_file_path=csv_file,
            batch_size=20,      # Download 20 video per batch
            start_from=0        # Mulai dari index 0
        )
        
        if results is not None:
            print("🎉 Download completed successfully!")
        else:
            print("❌ Download failed. Check error messages above.")
    else:
        print("👋 Download cancelled.")

# TROUBLESHOOTING GUIDE
"""
🔧 TROUBLESHOOTING:

1. LOGIN ERROR:
   - Login ke Instagram di browser Chrome
   - Refresh halaman Instagram
   - Jalankan script lagi

2. RATE LIMIT:
   - Kurangi batch_size ke 10-15
   - Tingkatkan sleep time di script
   - Gunakan VPN dengan IP berbeda

3. SLOW DOWNLOAD:
   - Kurangi batch_size
   - Pastikan koneksi internet stabil
   - Close aplikasi lain yang pakai bandwidth

4. RESUME DOWNLOAD:
   - Lihat batch terakhir yang berhasil
   - Set start_from ke index yang sesuai
   
5. BROWSER COOKIE ISSUES:
   - Coba browser lain: ubah 'chrome' ke 'firefox' atau 'safari'
   - Clear browser cache dan login ulang
"""

🔧 SETUP INSTRUCTIONS:
1. Make sure you're logged into Instagram in Chrome browser
2. Install required packages: pip install yt-dlp pandas
3. The script will use cookies from Chrome automatically



Ready to start downloading? (y/n):  y


🚀 SIMPLE INSTAGRAM VIDEO DOWNLOADER
📖 Reading CSV: C:\Users\Aditya P J\Documents\Competition\Satria Data 2025\data\data_train_clean.csv
📊 Found columns: ['id', 'video', 'emotion']
📈 Total rows: 795
🎯 Sample data:
   id                                              video   emotion
0   1  https://www.instagram.com/reel/DNKcHgdA-d1/?ig...  Surprise
1   2  https://www.instagram.com/reel/DNHwrh2gnBm/?ig...  Surprise
2   3  https://www.instagram.com/reel/DM7QsjnRCoa/?ig...  Surprise
3   4  https://www.instagram.com/reel/DNBBEt6Paxj/?ig...  Surprise
4   5  https://www.instagram.com/reel/DMz13fQzZsN/?ig...     Proud

🎯 Processing 795 rows starting from index 0

🔄 Processing Batch 1 - 20 videos
📥 [1/20] Downloading video 1 (Surprise)
[Instagram] Extracting URL: https://www.instagram.com/reel/DNKcHgdA-d1/?igsh=MWFxdnhjbzh1YmdhYw%3D%3D
[Instagram] DNKcHgdA-d1: Setting up session
[Instagram] DNKcHgdA-d1: Downloading JSON metadata
[info] DNKcHgdA-d1: Downloading 1 format(s): 1
[download] Sleeping 4.

[info] DK_a9aazNPI: Downloading 1 format(s): 1
[download] Sleeping 3.42 seconds ...
[download] Destination: downloaded_videos\174_Trust.mp4
[download] 100% of   12.81MiB in 00:00:02 at 6.32MiB/s   
✅ Success: 174_Trust.mp4 (12.8 MB)
⏳ Waiting 3 seconds...
📥 [15/20] Downloading video 175 (Surprise)
[Instagram] Extracting URL: https://www.instagram.com/reel/DGmzQ0Nyjlo/?igsh=MW5scHM5ZzNpNGlrOQ==
[Instagram] DGmzQ0Nyjlo: Setting up session
[Instagram] DGmzQ0Nyjlo: Downloading JSON metadata
[info] DGmzQ0Nyjlo: Downloading 1 format(s): 8
[download] Sleeping 2.98 seconds ...
[download] Destination: downloaded_videos\175_Surprise.mp4
[download] 100% of   46.02MiB in 00:00:03 at 11.62MiB/s  
✅ Success: 175_Surprise.mp4 (46.0 MB)
⏳ Waiting 3 seconds...
📥 [16/20] Downloading video 176 (Surprise)
[Instagram] Extracting URL: https://www.instagram.com/reel/DB7wNtHygCF/?igsh=a2ozNjY3Y21wOTVs
[Instagram] DB7wNtHygCF: Setting up session
[Instagram] DB7wNtHygCF: Downloading JSON metadata
[info] DB7wNtH

ERROR: [GoogleDrive] 1-n-P6UQqIBYoTzxtEze3qKSj8A8hYlr7: You don't have permission to access this video.


...

[info] C4ijfChSaGd: Downloading 1 format(s): 2
[download] Sleeping 2.27 seconds ...
[download] Destination: downloaded_videos\788_Surprise.mp4
[download] 100% of   17.24MiB in 00:00:02 at 6.75MiB/s   
✅ Success: 788_Surprise.mp4 (17.2 MB)
⏳ Waiting 3 seconds...

📊 Batch 39 Summary:
   ✅ Downloaded: 20/20 videos
   📊 Success rate: 100.0%
   💾 Total size: 253.9 MB
   💾 Saved to: batch_videos_39.csv

🔄 Processing Batch 40 - 15 videos
📥 [1/15] Downloading video 789 (Proud)
[Instagram] Extracting URL: https://www.instagram.com/reel/DBSQKPcuq4V/?igsh=MTl3YWVpZWVkbmlxOA%3D%3D
[Instagram] DBSQKPcuq4V: Setting up session
[Instagram] DBSQKPcuq4V: Downloading JSON metadata
[info] DBSQKPcuq4V: Downloading 1 format(s): 3
[download] Sleeping 3.49 seconds ...
[download] Destination: downloaded_videos\789_Proud.mp4
[download] 100% of    7.86MiB in 00:00:01 at 6.35MiB/s   
✅ Success: 789_Proud.mp4 (7.9 MB)
⏳ Waiting 3 seconds...
📥 [2/15] Downloading video 790 (Surprise)
[Instagram] Extracting URL: http

[Instagram] DMVRWpmTqUC: Downloading JSON metadata


ERROR: [Instagram] DMVRWpmTqUC: Instagram sent an empty media response. Check if this post is accessible in your browser without being logged-in. If it is not, then use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Otherwise, if the post is accessible in browser without being logged-in, please report this issue on  https://github.com/yt-dlp/yt-dlp/issues?q= , filling out the appropriate issue template. Confirm you are on the latest version using  yt-dlp -U


❌ Error downloading video 796: ERROR: [Instagram] DMVRWpmTqUC: Instagram sent an empty media response. Check if this post is accessible in your browser without being logged-in. If it is not, then use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Otherwise, if the post is accessible in browser without being logged-in, please report this issue on  https://github.com/yt-dlp/yt-dlp/issues?q= , filling out the appropriate issue template. Confirm you are on the latest version using  yt-dlp -U
❌ Failed: Could not download video 796
⏳ Waiting 3 seconds...
📥 [9/15] Downloading video 797 (Proud)
[Instagram] Extracting URL: https://www.instagram.com/reel/DDqXKnISXIM/?igsh=MThhNGtkaGZ5Y3Vxcg==
[Instagram] DDqXKnISXIM: Setting up session
[Instagram] DDqXKnISXIM: Downloading JSON metadata
[info] DDqXKnISXIM: Downloading 1 format(s): 7
[download] Sleeping 4.73 seconds ...
[d

"\n🔧 TROUBLESHOOTING:\n\n1. LOGIN ERROR:\n   - Login ke Instagram di browser Chrome\n   - Refresh halaman Instagram\n   - Jalankan script lagi\n\n2. RATE LIMIT:\n   - Kurangi batch_size ke 10-15\n   - Tingkatkan sleep time di script\n   - Gunakan VPN dengan IP berbeda\n\n3. SLOW DOWNLOAD:\n   - Kurangi batch_size\n   - Pastikan koneksi internet stabil\n   - Close aplikasi lain yang pakai bandwidth\n\n4. RESUME DOWNLOAD:\n   - Lihat batch terakhir yang berhasil\n   - Set start_from ke index yang sesuai\n   \n5. BROWSER COOKIE ISSUES:\n   - Coba browser lain: ubah 'chrome' ke 'firefox' atau 'safari'\n   - Clear browser cache dan login ulang\n"